## Analyse du titre de l'article en fonction des noms d'entreprises présents dans le dictionnaire

In [5]:
import pandas as pd
import torch
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import yfinance as yf
import tensorflow as tf
from datetime import datetime

# 1. Chargement et nettoyage
df = pd.read_csv('Data/cac40_news.csv')
df['publish_time'] = pd.to_datetime(df['providerPublishTime'], unit='s')
df = df[['title', 'publish_time']].dropna()

# 2. Dictionnaire des 40 tickers du CAC 40
ticker_dict = {
    "Accor": "AC.PA", "Air Liquide": "AI.PA", "Airbus": "AIR.PA",
    "ArcelorMittal": "MT.AS", "Axa": "CS.PA", "BNP Paribas": "BNP.PA",
    "Bouygues": "EN.PA", "Capgemini": "CAP.PA", "Carrefour": "CA.PA",
    "Crédit Agricole": "ACA.PA", "Danone": "BN.PA", "Dassault Systèmes": "DSY.PA",
    "Edenred": "EDEN.PA", "Engie": "ENGI.PA", "EssilorLuxottica": "EL.PA",
    "Eurofins Scientific": "ERF.PA", "Hermès": "RMS.PA", "Kering": "KER.PA",
    "L'Oréal": "OR.PA", "Legrand": "LR.PA", "LVMH": "MC.PA",
    "Michelin": "ML.PA", "Orange": "ORA.PA", "Pernod Ricard": "RI.PA",
    "Publicis": "PUB.PA", "Renault": "RNO.PA", "Safran": "SAF.PA",
    "Saint-Gobain": "SGO.PA", "Sanofi": "SAN.PA", "Schneider Electric": "SU.PA",
    "Société Générale": "GLE.PA", "Stellantis": "STLAP.PA", "STMicroelectronics": "STMPA.PA",
    "Teleperformance": "TEP.PA", "Thales": "HO.PA", "TotalEnergies": "TTE.PA",
    "Unibail-Rodamco-Westfield": "URW.PA", "Veolia": "VIE.PA", "Vinci": "DG.PA",
    "Vivendi": "VIV.PA"
}
def extract_tickers(title):
    txt = title.lower()
    return [t for name,t in ticker_dict.items() if name.lower() in txt]

df['tickers'] = df['title'].apply(extract_tickers)
df = df.explode('tickers').dropna(subset=['tickers'])

# 3. Chargement du tokenizer et du modèle FinBERT
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
# from_pt=True convertit les poids PyTorch vers un modèle TF
model_tf  = TFAutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    from_pt=True  # conversion PyTorch → TF
)
model_tf.trainable = False

# 4. Fonction sentiment sans pipeline, en PyTorch pur
def get_sentiment_score_tf(text):
    inputs  = tokenizer(text, return_tensors="tf", truncation=True, max_length=512)
    outputs = model_tf(**inputs)
    scores  = tf.nn.softmax(outputs.logits, axis=-1)[0]
    # indices 1 = positive, 2 = negative
    return float(scores[1] - scores[2])

df['sentiment'] = df['title'].apply(get_sentiment_score_tf)

# 5. Agrégation quotidienne
df['date'] = df['publish_time'].dt.date
agg = df.groupby(['tickers','date']).agg(
    avg_sentiment=('sentiment','mean'),
    article_count=('sentiment','size')
).reset_index()

# 6. Génération des signaux
TH_POS, TH_NEG, MIN_ART = 0.10, -0.10, 3
signals = []
for _, r in agg.iterrows():
    sig = 'HOLD'
    if r.article_count >= MIN_ART:
        if r.avg_sentiment >= TH_POS: sig = 'BUY'
        elif r.avg_sentiment <= TH_NEG: sig = 'SELL'
    signals.append((r.date, r.tickers, r.avg_sentiment, r.article_count, sig))

signals_df = pd.DataFrame(signals, columns=['date','ticker','avg_sentiment','count','signal'])
print(signals_df.head())

# Diagnostic
print("→ Signaux générés :", len(signals_df))
print("→ Tickers détectés :", signals_df['ticker'].unique().tolist())

# Backtest s'il y a au moins un ticker
tickers = signals_df['ticker'].unique().tolist()
if tickers:
    prices = yf.download(
        tickers,
        start = signals_df['date'].min(),
        end   = datetime.today().date(),
        progress=False
    )
    if prices.empty:
        print("yfinance n'a renvoyé aucune donnée pour :", tickers)
    else:
       rets = prices['Adj Close'].pct_change().reset_index()
else:
    print(" Aucun ticker valide trouvé dans les signaux.")

2025-04-27 01:40:51.640146: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertForSequenceClassification: ['bert.embeddings.position_ids']
- This IS expected if you are initializing TFBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertForSeque

Empty DataFrame
Columns: [date, ticker, avg_sentiment, count, signal]
Index: []
→ Signaux générés : 0
→ Tickers détectés : []
 Aucun ticker valide trouvé dans les signaux.


## Analyse du contenu de l'article en fonction des tendances politiques

In [10]:
import pandas as pd
from newspaper import Article
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datetime import datetime

# 1. chargement des données
df = pd.read_csv('Data/cac40_news.csv')
df = df[['title','link','providerPublishTime']].dropna()
df = df.drop_duplicates(subset='link')
df['publish_time'] = pd.to_datetime(df['providerPublishTime'], unit='s')




# 2. Configuration des mappers
# 2a. Sociétés du CAC40
ticker_dict = {
    "Accor": "AC.PA", "Air Liquide": "AI.PA", "Airbus": "AIR.PA",
    "ArcelorMittal": "MT.AS", "Axa": "CS.PA", "BNP Paribas": "BNP.PA",
    "Bouygues": "EN.PA", "Capgemini": "CAP.PA", "Carrefour": "CA.PA",
    "Crédit Agricole": "ACA.PA", "Danone": "BN.PA", "Dassault Systèmes": "DSY.PA",
    "Edenred": "EDEN.PA", "Engie": "ENGI.PA", "EssilorLuxottica": "EL.PA",
    "Eurofins Scientific": "ERF.PA", "Hermès": "RMS.PA", "Kering": "KER.PA",
    "L'Oréal": "OR.PA", "Legrand": "LR.PA", "LVMH": "MC.PA",
    "Michelin": "ML.PA", "Orange": "ORA.PA", "Pernod Ricard": "RI.PA",
    "Publicis": "PUB.PA", "Renault": "RNO.PA", "Safran": "SAF.PA",
    "Saint-Gobain": "SGO.PA", "Sanofi": "SAN.PA", "Schneider Electric": "SU.PA",
    "Société Générale": "GLE.PA", "Stellantis": "STLAP.PA", "STMicroelectronics": "STMPA.PA",
    "Teleperformance": "TEP.PA", "Thales": "HO.PA", "TotalEnergies": "TTE.PA",
    "Unibail-Rodamco-Westfield": "URW.PA", "Veolia": "VIE.PA", "Vinci": "DG.PA",
    "Vivendi": "VIV.PA"
}

# 2b. Personnalités / Entreprises non-CAC40
person_to_ticker = {
    "Elon Musk": "TSLA",
    "Jeff Bezos": "AMZN",
    "Mark Zuckerberg": "META",
    "Larry Page": "GOOGL",
    "Sergey Brin": "GOOGL",
    "Tim Cook": "AAPL",
    "Sundar Pichai": "GOOGL",
    "Bill Gates": "MSFT",
}

# Combine les deux mappers
entity2ticker = {**ticker_dict, **person_to_ticker}





# 3. NLP setup
# 3a. Récupérer le texte complet
def fetch_full_text(url):
    art = Article(url, language='en')  # ou 'fr' si la majorité est en français
    art.download(); art.parse()
    return art.text

# 3b. Chargement spaCy pour la NER
nlp = spacy.load('en_core_web_sm')  # ou fr_core_news_sm si FR

# 3c. Chargement FinBERT (PyTorch)
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model     = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

def sentiment_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    # indices : 0=neutral, 1=positive, 2=negative
    return float(probs[1] - probs[2])







# 4. Pipeline par article - On se limite aux entités du dictionnaire défini plus haut
results = []

for idx, row in df.iterrows():
    title, url = row['title'], row['link']
    # 4.1 Récupère le texte
    try:
        text = fetch_full_text(url)
    except Exception as e:
        print(f"Échec fetch pour {url}: {e}")
        continue

    # 4.2 Extrait les entités
    doc = nlp(text)
    # on ne garde que ORG et PERSON
    entities = set(ent.text for ent in doc.ents if ent.label_ in ("ORG","PERSON"))
    # on ne considère que ceux qu'on sait mapper
    tickers = {entity2ticker[e]: e for e in entities if e in entity2ticker}

    if not tickers:
        continue

    # 4.3 Scoring phrase par phrase
    sentences = [sent.text for sent in doc.sents]
    scores_by_ticker = {tkr: [] for tkr in tickers}
    for sent in sentences:
        for tkr, ent in tickers.items():
            if ent.lower() in sent.lower():
                sc = sentiment_score(sent)
                scores_by_ticker[tkr].append(sc)

    # 4.4 Agrégation et signal
    for tkr, scores in scores_by_ticker.items():
        if not scores:
            continue
        avg = sum(scores) / len(scores)
        if avg >  0.1: signal = "BUY"
        elif avg < -0.1: signal = "SELL"
        else:           signal = "HOLD"

        results.append({
            "date":        row['publish_time'].date(),
            "ticker":      tkr,
            "entity":      tickers[tkr],
            "avg_sent":    round(avg,3),
            "n_phrases":   len(scores),
            "signal":      signal,
            "title":       title,
            "url":         url
        })



# 5. Résultats dans un DataFrame
res_df = pd.DataFrame(results)
print(res_df.head(20))


         date ticker         entity  avg_sent  n_phrases signal  \
0  2025-03-11   TSLA      Elon Musk    -0.253          2   SELL   
1  2025-03-11   TSLA      Elon Musk    -0.217          4   SELL   
2  2025-03-11   TSLA      Elon Musk    -0.484          3   SELL   
3  2025-03-11  GOOGL  Sundar Pichai    -0.860          1   SELL   

                                               title  \
0  National wildlife refuges in Florida hit by Tr...   
1  Rocket builder vs NASA astronaut: Musk, Sen. K...   
2  Trump says he'll buy a Tesla to show support f...   
3  IonQ Could Be a Quantum Computing Powerhouse, ...   

                                                 url  
0  https://finance.yahoo.com/m/9e6ea28d-ceb3-32ba...  
1  https://finance.yahoo.com/m/bc2ea7c9-4efb-39c3...  
2  https://finance.yahoo.com/news/trump-says-hell...  
3  https://finance.yahoo.com/m/505da054-1cc2-3510...  


------------------------------

## TESTS

In [9]:
import pandas as pd
from newspaper import Article
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1. Préparation générale ---
df = pd.read_csv('Data/cac40_news.csv')
df = df[['title','link','providerPublishTime']].dropna()
df = df.drop_duplicates(subset='link')
df['publish_time'] = pd.to_datetime(df['providerPublishTime'], unit='s')

# --- 2. Chargement des outils NLP ---
# Texte complet
def fetch_full_text(url):
    art = Article(url, language='en')
    art.download(); art.parse()
    return art.text

# NER
nlp = spacy.load("en_core_web_sm")  # ou fr_core_news_sm

# FinBERT PyTorch
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model     = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

def sentiment_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    return float(probs[1] - probs[2])  # + = positif, – = négatif

# --- 3. Pipeline d’analyse “tout-ORG/PERSON” ---
records = []

for _, row in df.iterrows():
    try:
        text = fetch_full_text(row['link'])
    except:
        continue

    doc = nlp(text)
    # repère TOUTES les entités ORG et PERSON, sans filtrage
    ents = [ent.text.strip() for ent in doc.ents if ent.label_ in ("ORG","PERSON")]
    if not ents:
        continue

    # pré-initialise une liste de scores par entité unique
    scores = {e: [] for e in set(ents)}

    # boucle phrase par phrase
    for sent in doc.sents:
        s_text = sent.text
        score = sentiment_score(s_text)
        # ajoute le score à chaque entité apparaissant dans la phrase
        for ent in scores:
            if ent in s_text:
                scores[ent].append(score)

    # agrège et crée un signal
    for ent, lst in scores.items():
        if not lst:
            continue
        avg = sum(lst) / len(lst)
        if   avg >  0.1: sig = "BUY"
        elif avg < -0.1: sig = "SELL"
        else:             sig = "HOLD"

        records.append({
            "date":      row['publish_time'].date(),
            "entity":    ent,
            "avg_sent":  round(avg, 3),
            "n_phrases": len(lst),
            "signal":    sig,
            "title":     row['title'],
            "url":       row['link']
        })

# --- 4. Résultat final ---
res_df = pd.DataFrame(records)
print(res_df.sort_values(['date','entity']).head(20))

           date                                             entity  avg_sent  \
127  2025-03-11                                               ABRA    -0.933   
28   2025-03-11                                                 AI    -0.176   
99   2025-03-11                                                 AP    -0.132   
183  2025-03-11                                                 AP    -0.413   
58   2025-03-11                                                AWS    -0.303   
117  2025-03-11                                         AbraSilver    -0.579   
129  2025-03-11                           AbraSilver Resource Corp    -0.929   
182  2025-03-11                                         Alex Jones    -0.819   
209  2025-03-11                                           Alphabet    -0.807   
64   2025-03-11                                             Amazon     0.090   
65   2025-03-11                                Amazon Web Services    -0.038   
30   2025-03-11                         